In [22]:
import duckdb
from pathlib import Path

In [23]:
BASE = Path.cwd().parent.parent
DATA_DIR = BASE / "data" / "options"
OUT_DIR = BASE / "output"

STATS_GLOB   = str(DATA_DIR / "statistics" / "*.parquet")

In [24]:
con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")

In [25]:
con.sql(f"""
SELECT
    COUNT(*)                                        AS n_rows,
    COUNT(DISTINCT instrument_id)                   AS n_contracts,
    COUNT(ts_ref)                                   AS n_with_ts_ref,
    MIN(ts_event)                                   AS earliest,
    MAX(ts_event)                                   AS latest
FROM read_parquet('{STATS_GLOB}')
""").df()

,n_rows,n_contracts,n_with_ts_ref,earliest,latest
0,99307437,89240,29773803,2023-03-20 09:58:00.021000+00:00,2026-06-26 21:00:00.127000+00:00


**Instrument classes**

Same instrument classes as in definitions (C, P, T, M).
- Symbols containing `OMC` are **Call** (30.4M)
- Symbols containing `OMP` are **Put** (30.3M)
- Rest is categorized with **Other** (38.6M)

In [26]:
con.sql(f"""
    SELECT 
        CASE 
            WHEN symbol LIKE '%_OMC%' THEN 'Call'
            WHEN symbol LIKE '%_OMP%' THEN 'Put'
            ELSE 'Other'
        END as instrument_type,
        COUNT(*) as count
    FROM read_parquet('{STATS_GLOB}')
    GROUP BY instrument_type
    ORDER BY count DESC
""").df()

,instrument_type,count
0,Other,38643103
1,Call,30397066
2,Put,30267268


**Coverage: 2023-03-20 to 2026-06-27** (3 years).

In [27]:
con.sql(f"""
    SELECT 
        MIN(ts_event) as earliest,
        MAX(ts_event) as latest,
        COUNT(*) as total_rows
    FROM read_parquet('{STATS_GLOB}')
    WHERE symbol LIKE '%_OMC%'
       OR symbol LIKE '%_OMP%'
""").df()

,earliest,latest,total_rows
0,2023-03-20 09:58:00.021000+00:00,2026-06-26 21:00:00.127000+00:00,60664334


**Final vs. preliminary**

**18,371,187** out of **60,664,334** rows have `ts_ref` populated (30%)

`ts_ref` is the timestamp of the record, end of trading session (15:00 CET). 
- Populated = final settlement
- Null = preliminary or non-settlement stat type.

In [28]:
con.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(ts_ref) as non_null
    FROM read_parquet('{STATS_GLOB}')
    WHERE symbol LIKE '%_OMC%'
       OR symbol LIKE '%_OMP%'
""").df()

,total_rows,non_null
0,60664334,18371187


**Statistics types**

After applying `stat_type IN (3, 14)`, `ts_ref IS NOT NULL` and the
`stat_flags` filter for final settlements:

- **stat_type 3:** 5,807,828 final settlement prices
- **stat_type 14:** 5,812,246 implied volatilities

Total: 11,620,074 records. The counts differ by 4,418, the preliminary
settlement prices removed by the `stat_flags` filter.

In [45]:
con.sql(f"""
    SELECT 
        stat_type,
        COUNT(*)                                              AS total,
        COUNT(ts_ref)                                         AS ts_ref_populated,
        COUNT(CASE WHEN stat_type != 3 OR stat_flags = 1
                   THEN ts_ref END)                           AS final
    FROM read_parquet('{STATS_GLOB}')
    WHERE (symbol LIKE '%_OMC%' OR symbol LIKE '%_OMP%')
      AND stat_type IN (3, 14)
    GROUP BY stat_type
    ORDER BY stat_type
""").df()

,stat_type,total,ts_ref_populated,final
0,3,11612442,5812246,5807828
1,14,5812246,5812246,5812246


**Duplicate contract-days**

A contract-day can carry more than one record for the same statistic, where ICE revised the published value. Never more than two.

**392,904** contract days with duplicates

In [35]:
con.sql(f"""
    SELECT COUNT(*) AS contract_days_with_duplicates
    FROM (
        SELECT instrument_id, CAST(ts_ref AS DATE) AS trading_date, stat_type
        FROM read_parquet('{STATS_GLOB}')
        WHERE (symbol LIKE '%_OMC%' OR symbol LIKE '%_OMP%')
          AND stat_type IN (3, 14)
          AND ts_ref IS NOT NULL
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").df()

,contract_days_with_duplicates
0,392904


Of those, the cases where the two records carry genuinely **different prices**. 

**4,128** (1%) duplicates with different prices, the rest are identical republication.

In [36]:
con.sql(f"""
    SELECT COUNT(*) AS duplicates_with_different_prices
    FROM (
        SELECT instrument_id, CAST(ts_ref AS DATE) AS trading_date, stat_type,
               MIN(price) AS min_price, MAX(price) AS max_price
        FROM read_parquet('{STATS_GLOB}')
        WHERE (symbol LIKE '%_OMC%' OR symbol LIKE '%_OMP%')
          AND stat_type IN (3, 14)
          AND ts_ref IS NOT NULL
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
    WHERE min_price != max_price
""").df()

,duplicates_with_different_prices
0,4128


Revisions cluster on a few dates, where ICE recalculated many contracts at once.

In [32]:
con.sql(f"""
    SELECT trading_date, stat_type, COUNT(*) AS n_revisions
    FROM (
        SELECT CAST(ts_ref AS DATE) AS trading_date, stat_type, instrument_id,
               MIN(price) AS min_price, MAX(price) AS max_price
        FROM read_parquet('{STATS_GLOB}')
        WHERE (symbol LIKE '%_OMC%' OR symbol LIKE '%_OMP%')
          AND stat_type IN (3, 14)
          AND ts_ref IS NOT NULL
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1 AND MIN(price) != MAX(price)
    )
    GROUP BY 1, 2
    ORDER BY n_revisions DESC
    LIMIT 10
""").df()

,trading_date,stat_type,n_revisions
0,2026-05-04,3,626
1,2026-05-04,14,626
2,2023-12-04,14,526
3,2023-12-04,3,526
4,2026-03-30,3,442
5,2026-03-30,14,442
6,2025-10-22,3,288
7,2025-10-22,14,288
8,2025-10-03,14,108
9,2025-10-03,3,104


11,620,074 clean rows = 5,807,828 final settlement prices + 5,812,246 implied volatilities

4,418 preliminary settlement prices removed via the stat_flags filter.

Only $20,252$ unique symbols

In [44]:
con.sql(f"""
    SELECT
        COUNT(*)                       AS total_rows,
        COUNT(DISTINCT instrument_id)  AS unique_contracts
    FROM (
        SELECT instrument_id,
               ROW_NUMBER() OVER (
                   PARTITION BY instrument_id, CAST(ts_ref AS DATE), stat_type
                   ORDER BY sequence DESC
               ) AS rn
        FROM read_parquet('{STATS_GLOB}')
        WHERE (symbol LIKE '%_OMC%' OR symbol LIKE '%_OMP%')
          AND stat_type IN (3, 14)
          AND ts_ref IS NOT NULL
          AND (stat_type != 3 OR stat_flags = 1)
    )
    WHERE rn = 1
""").df()

,total_rows,unique_contracts
0,11231588,20252
